In [3]:
import pandas as pd
import numpy as np

In [4]:
df_products = pd.read_csv('../data/fda_cleaned_datafiles/Products.csv')
df_apps = pd.read_csv('../data/fda_cleaned_datafiles/Applications.csv')
prod_apps = pd.merge(df_products, df_apps, on='ApplNo', how='right').drop_duplicates(['DrugName'])
prod_apps['DrugName'] = prod_apps['DrugName'].str.lower()
prod_apps['ActiveIngredient'] = prod_apps['ActiveIngredient'].str.lower()
prod_apps

,ApplNo,ProductNo,Form,Strength,ReferenceDrug,DrugName,ActiveIngredient,ReferenceStandard,ApplType,ApplPublicNotes,SponsorName
0,4,4.0,SOLUTION/DROPS;OPHTHALMIC,1%,0.0,paredrine,hydroxyamphetamine hydrobromide,0.0,NDA,NaN,PHARMICS
1,159,1.0,TABLET;ORAL,500MG,0.0,sulfapyridine,sulfapyridine,0.0,NDA,NaN,LILLY
2,552,1.0,INJECTABLE;INJECTION,"20,000 UNITS/ML",0.0,liquaemin sodium,heparin sodium,0.0,NDA,NaN,ASPEN GLOBAL INC
7,552,7.0,INJECTABLE;INJECTION,100 UNITS/ML,0.0,liquaemin lock flush,heparin sodium,0.0,NDA,NaN,ASPEN GLOBAL INC
8,552,8.0,INJECTABLE;INJECTION,"1,000 UNITS/ML",0.0,heparin sodium,heparin sodium,0.0,NDA,NaN,ASPEN GLOBAL INC
...,...,...,...,...,...,...,...,...,...,...,...
50823,761456,1.0,INJECTABLE;SUBCUTANEOUS,60MG/ML,0.0,boncresa,denosumab,0.0,BLA,NaN,AMNEAL PHARMS LLC
50824,761457,1.0,INJECTABLE;SUBCUTANEOUS,120MG/1.7ML (70MG/ML),0.0,oziltus,denosumab,0.0,BLA,NaN,AMNEAL PHARMS LLC
50825,761458,1.0,INJECTION,100MG,0.0,exdensur,depemokimab-ulaa,0.0,BLA,NaN,GLAXOSMITHKLINE LLC
50827,761467,1.0,INJECTABLE;INJECTION,395MG/4800UNITS(2.4ML),0.0,keytruda qlex,pembrolizumab;berahyaluronidase alfa-pmph,0.0,BLA,NaN,MERCK SHARP DOHME


In [9]:
# this is for drug dashboard project, ignore
num_sponsors = prod_apps.groupby('ActiveIngredient')['SponsorName'].nunique().reset_index()
num_sponsors
num_sponsors['single_source'] = np.where(num_sponsors['SponsorName'] == 1, 1, 0)

prod_apps_source = pd.merge(prod_apps, num_sponsors, on='ActiveIngredient')
prod_apps_source = prod_apps.groupby('ActiveIngredient')['SponsorName'].nunique().reset_index()
num_sponsors
num_sponsors['single_source'] = np.where(num_sponsors['SponsorName'] == 1, 1, 0)

prod_apps_source = pd.merge(prod_apps, num_sponsors, on='ActiveIngredient')
prod_apps_source = prod_apps_source[['ApplNo','DrugName','SponsorName_x','single_source','ActiveIngredient']]
prod_apps_source

################
ndc_dir = pd.read_csv('../data/final_combined_2013_2025_for_supabase.csv')
prod_apps_source['ApplNo'] = prod_apps_source['ApplNo'].astype(str)
ndc_dir['APPLICATIONNUMBER'] = ndc_dir['APPLICATIONNUMBER'].str.replace(r'\D','',regex=True)

In [8]:
ndc_dir

,PROPRIETARYNAME,NONPROPRIETARYNAME,DOSAGEFORMNAME,ROUTENAME,MARKETINGCATEGORYNAME,APPLICATIONNUMBER,LABELERNAME,SUBSTANCENAME,ACTIVE_NUMERATOR_STRENGTH,ACTIVE_INGRED_UNIT,DEASCHEDULE,PRODUCTNDC,Year,key
0,Amyvid,Florbetapir F 18,"INJECTION, SOLUTION",INTRAVENOUS,NDA,202008,Eli Lilly and Company,FLORBETAPIR F-18,51,mCi/mL,NaN,0002-1200,2013,0
1,Quinidine Gluconate,Quinidine Gluconate,SOLUTION,INTRAVENOUS,NDA,007529,Eli Lilly and Company,QUINIDINE GLUCONATE,80,mg/mL,NaN,0002-1407,2013,1
2,AXIRON,testosterone,SOLUTION,TOPICAL,NDA,022504,Eli Lilly and Company,TESTOSTERONE,30,mg/1.5mL,CIII,0002-1975,2013,2
3,Prozac,Fluoxetine hydrochloride,"CAPSULE, DELAYED RELEASE",ORAL,NDA,021235,Eli Lilly and Company,FLUOXETINE HYDROCHLORIDE,90,mg/1,NaN,0002-3004,2013,3
4,Strattera,Atomoxetine hydrochloride,CAPSULE,ORAL,NDA,021411,Eli Lilly and Company,ATOMOXETINE HYDROCHLORIDE,10,mg/1,NaN,0002-3227,2013,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
343068,Valerin,"Valerian Root, Passiflora, Magnesium Carbonate",TABLET,ORAL,UNAPPROVED HOMEOPATHIC,NaN,Wonder Laboratories,MAGNESIUM CARBONATE; PASSIFLORA INCARNATA FLOW...,1; 3; 6,[hp_X]/1; [hp_X]/1; [hp_X]/1,NaN,99528-606,2025,343068
343069,Valerin,"Valerian Root, Passiflora, Magnesium Carbonate",TABLET,ORAL,UNAPPROVED HOMEOPATHIC,NaN,Wonder Laboratories,MAGNESIUM CARBONATE; PASSIFLORA INCARNATA FLOW...,1; 3; 6,[hp_X]/1; [hp_X]/1; [hp_X]/1,NaN,99528-606,2025,343069
343070,Valerin,"Valerian Root, Passiflora, Magnesium Carbonate",TABLET,ORAL,UNAPPROVED HOMEOPATHIC,NaN,Wonder Laboratories,MAGNESIUM CARBONATE; PASSIFLORA INCARNATA FLOW...,1; 3; 6,[hp_X]/1; [hp_X]/1; [hp_X]/1,NaN,99528-606,2025,343070
343071,Valerin,"Valerian Root, Passiflora, Magnesium Carbonate",TABLET,ORAL,UNAPPROVED HOMEOPATHIC,NaN,Wonder Laboratories,MAGNESIUM CARBONATE; PASSIFLORA INCARNATA FLOW...,1; 3; 6,[hp_X]/1; [hp_X]/1; [hp_X]/1,NaN,99528-606,2025,343071


In [12]:
fda_ndc_prod_apps = pd.merge(prod_apps_source, ndc_dir[['PROPRIETARYNAME','APPLICATIONNUMBER','PRODUCTNDC','ROUTENAME','SUBSTANCENAME','LABELERNAME']], left_on='ApplNo', right_on='APPLICATIONNUMBER', how='right').drop_duplicates()
fda_ndc_prod_apps

,ApplNo,DrugName,SponsorName_x,single_source,ActiveIngredient,PROPRIETARYNAME,APPLICATIONNUMBER,PRODUCTNDC,ROUTENAME,SUBSTANCENAME,LABELERNAME
0,202008,amyvid,AVID RADIOPHARMS INC,1.0,florbetapir f-18,Amyvid,202008,0002-1200,INTRAVENOUS,FLORBETAPIR F-18,Eli Lilly and Company
1,NaN,NaN,NaN,NaN,NaN,Quinidine Gluconate,007529,0002-1407,INTRAVENOUS,QUINIDINE GLUCONATE,Eli Lilly and Company
2,NaN,NaN,NaN,NaN,NaN,AXIRON,022504,0002-1975,TOPICAL,TESTOSTERONE,Eli Lilly and Company
3,NaN,NaN,NaN,NaN,NaN,Prozac,021235,0002-3004,ORAL,FLUOXETINE HYDROCHLORIDE,Eli Lilly and Company
4,NaN,NaN,NaN,NaN,NaN,Strattera,021411,0002-3227,ORAL,ATOMOXETINE HYDROCHLORIDE,Eli Lilly and Company
...,...,...,...,...,...,...,...,...,...,...,...
343835,NaN,NaN,NaN,NaN,NaN,Zyclara,022483,99207-271,TOPICAL,IMIQUIMOD,"Bausch Health US, LLC"
343837,NaN,NaN,NaN,NaN,NaN,Zyclara,022483,99207-276,TOPICAL,IMIQUIMOD,"Bausch Health US, LLC"
343839,NaN,NaN,NaN,NaN,NaN,ZIANA,050802,99207-300,TOPICAL,CLINDAMYCIN PHOSPHATE; TRETINOIN,"Bausch Health US, LLC"
343844,204153,luzu,BAUSCH,1.0,luliconazole,Luzu,204153,99207-850,TOPICAL,LULICONAZOLE,"Bausch Health US, LLC"


In [13]:
fda_ndc_prod_apps.to_csv('../data/NDC_FDA.csv', index=True)

In [14]:
fda_ndc_prod_apps['ROUTENAME'].unique()

array(['INTRAVENOUS', 'TOPICAL', 'ORAL', 'INTRAMUSCULAR; SUBCUTANEOUS',
       'INTRAVENOUS; SUBCUTANEOUS', 'SUBCUTANEOUS', 'INTRAMUSCULAR', nan,
       'PARENTERAL', 'INTRA-ARTICULAR; INTRAMUSCULAR',
       'INTRA-ARTICULAR; INTRALESIONAL', 'INTRAMUSCULAR; INTRAVENOUS',
       'OPHTHALMIC',
       'INTRALESIONAL; INTRAMUSCULAR; INTRASYNOVIAL; SOFT TISSUE',
       'INTRAMUSCULAR; INTRAVENOUS; SUBCONJUNCTIVAL',
       'INTRAVASCULAR; INTRAVENOUS', 'VAGINAL', 'INTRACAVERNOUS',
       'RESPIRATORY (INHALATION)', 'NASAL', 'URETERAL', 'INTRAVASCULAR',
       'INTRA-ARTERIAL; INTRAVENOUS', 'ORAL; RECTAL', 'CUTANEOUS',
       'INTRADERMAL; INTRAMUSCULAR', 'INTRAVITREAL', 'INTRAOCULAR',
       'URETHRAL', 'DENTAL',
       'INTRAMUSCULAR; INTRAPLEURAL; INTRATHECAL; INTRAVENOUS',
       'TRANSDERMAL', 'SUBLINGUAL', 'INTRAVESICAL', 'PERCUTANEOUS',
       'ORAL; RESPIRATORY (INHALATION)', 'ORAL; TOPICAL',
       'AURICULAR (OTIC)',
       'INTRA-ARTERIAL; INTRAMUSCULAR; INTRATHECAL; INTRAVENOUS',
